In [ ]:
import pandas as pd
import pickle 
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm import tqdm
import seaborn as sns 

In [ ]:
pixel_clustering_base_path = r"" # path to pixel clustering results

In [ ]:
with open(pixel_clustering_base_path + '/cluster_color_dict.pickle', 'rb') as f: 
    cluster_color_dict = pickle.load(f)

In [ ]:
with open(pixel_clustering_base_path + '/wsi_labeled.pickle', 'rb') as f:
    wsi_labeled = pickle.load(f)

In [ ]:
wsi_labeled[np.isnan(wsi_labeled)] = -1

In [ ]:
labels = wsi_labeled.astype(np.int32)
clusters = np.unique(labels[labels >= 0])
cluster_to_idx = {c: i for i, c in enumerate(clusters)}
n = len(clusters)

counts = np.zeros((n, n), dtype=np.int64)

label_idx = np.full_like(labels, -1)
for c, i in cluster_to_idx.items():
    label_idx[labels == c] = i

shifts = [
    (-1, -1), (-1, 0), (-1, 1), ( 0, -1), ( 0, 1),( 1, -1), ( 1, 0), ( 1, 1)
]

H, W = labels.shape

for dy, dx in tqdm(shifts):
    y0 = max(0, dy)
    y1 = min(H, H + dy)
    x0 = max(0, dx)
    x1 = min(W, W + dx)

    src = label_idx[y0:y1, x0:x1]
    dst = label_idx[y0-dy:y1-dy, x0-dx:x1-dx]

    valid = (src >= 0) & (dst >= 0)
    np.add.at(counts, (src[valid], dst[valid]), 1)


In [ ]:
with open(pixel_clustering_base_path + '/pixel_neighbors_count.pickle', 'wb') as f: 
    pickle.dump(counts, f)

In [ ]:
p = counts / counts.sum(axis=1, keepdims=True)

In [ ]:
p_no_self = p.copy()
np.fill_diagonal(p_no_self, 0)

row_sums = p_no_self.sum(axis=1, keepdims=True)
p_no_self = np.divide(
    p_no_self,
    row_sums,
    where=row_sums > 0
)

In [ ]:
import matplotlib as mpl

In [ ]:
mpl.rcParams["font.size"] = 15
plt.figure(figsize=(6, 5))
sns.heatmap(
    p_no_self,
    xticklabels=clusters,
    yticklabels=clusters,
    cmap='coolwarm',
    square=True
)

plt.xlabel('Neighbor Cluster')
plt.ylabel('Center Cluster')
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(6, 5))
sns.heatmap(
    p,
    xticklabels=clusters,
    yticklabels=clusters,
    cmap='coolwarm',
    square=True
)

plt.xlabel('Neighbor Cluster')
plt.ylabel('Center Cluster')
plt.tight_layout()
plt.show()
